# Visualizing regression results

In [1]:
import os
import numpy as np
import pandas as pd
from rpy2.robjects.lib.ggplot2 import theme
from scipy.linalg import lstsq

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import statsmodels.formula.api as smf
import warnings;warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../data/ckpts/rosen'

PARAMS_PATH = '../3-REGRESSION-MODELS/reports/complete-parameter-estimates.csv'

OUTPUTS_PATH = 'resids/reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(DATA_PATH)
len(fs)

1669

## Importing parameters

In [5]:
dfps = pd.read_csv(PARAMS_PATH)
dfps.head()

,param,beta,std,k,se
0,nx,0.272898,0.055406,4522,0.000824
1,ny,-0.042696,0.035685,4522,0.000531
2,self,-0.217913,1.140945,4522,0.016967


## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'AVAR'

In [8]:
# param_list = ['Intercept', 'nx', 'ny', 'self']
param_list = ['nx', 'ny', 'self']

In [39]:
df_ = []

In [40]:
for f in tqdm(fs):
    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id', # 'AVAR',
        'nx', 'ny']).to_pandas()

    df['self'] = (df['x_speaker'] == df['y_speaker']).astype(int)

    df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df['turn_delta'] = (df['y_turn_id'] - df['x_turn_id'])

    if ('CANDOR' in f.split('/')[-1]) or ('grace' in f.split('/')[-1]):
        df['turn_delta'].loc[df['self'] == 0] -= 1

    df['pred'] = (df['nx'] * dfps['beta'].loc[dfps['param'].isin(['nx'])].values[0]) + (df['ny'] * dfps['beta'].loc[dfps['param'].isin(['ny'])].values[0])

    df['resid'] = df[COEF_VAR] - df['pred']

    df_regression += [
        df[['self', 'turn_delta', 'resid']].groupby(by=['self', 'turn_delta']).agg('mean').reset_index()
    ]

    df_regression[-1]['std'] = df[['self', 'turn_delta', 'resid']].groupby(by=['self', 'turn_delta']).agg('std').reset_index()['resid']

    df_regression[-1]['k'] = df[['self', 'turn_delta', 'resid']].groupby(by=['self', 'turn_delta']).agg('count').reset_index()['resid']

    df_regression[-1]['se'] = df_regression[-1]['std'] / np.sqrt(df_regression[-1]['k'])

df_regression = pd.concat(df_regression, ignore_index=True)

  0%|          | 0/1669 [00:00<?, ?it/s]

In [41]:
# if not os.path.exists(os.path.join(OUTPUTS_PATH,'all.csv')):
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all.csv'),
#         index=False,
#         encoding='utf-8'
#     )
#
# else:
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all.csv'),
#         index=False,
#         header=False,
#         encoding='utf-8',
#         mode='a'
#     )

In [42]:
df0 = df_regression.groupby(by=['turn_delta', 'self']).agg('mean').reset_index()
df0.head()

,turn_delta,self,resid,std,k,se
0,1,0,0.086535,1.571404,40714.833333,0.016272
1,1,1,-0.645448,1.744177,22224.583333,0.018795
2,2,0,-0.012862,1.324760,20441.833333,0.016643
3,2,1,-0.561965,1.557404,42264.750000,0.014378
4,3,0,0.018942,1.485730,36374.583333,0.015964


## Plotly vis

In [43]:
import plotly.graph_objects as go

In [44]:
sel_1 = (df0['turn_delta'] <= 100) & (df0['turn_delta'] > 0) #& ((df0['turn_delta'] % 2) != 0)

In [45]:
fig = go.Figure()

In [46]:
sel = df0['self'] == 1
fig.add_trace(
    go.Scatter(
        x=df0['turn_delta'].loc[sel & sel_1].values,
        y = df0['resid'].loc[sel & sel_1].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange'
        )
    )
)

sel = df0['self'] == 0
fig.add_trace(
    go.Scatter(
        x=df0['turn_delta'].loc[sel & sel_1].values,
        y = df0['resid'].loc[sel & sel_1].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue'
        )
    )
)

fig.update_layout(template='xgridoff')